# TRAINING CNN FOR PNEUMONIA CLASSIFICATION

I will be using the same model for training models on different datasets (original xrays and transformed xrays); everything stays same besides the path of the dataset to be trained on. I coded this part using online youtube tutorials, a guide from medium, and I also have experience in working with CNNs from before (I knew about the model and training part, just had some issues with balancing the dataset). 

I trained the model locally on my mac, and used claude to see how I could train it on the GPU (which is why I had to download specific version of tensorflow)

## Importing Libraries

This is for loading the dataset, preprocessing where necessary, building the model, training and plotting results.

In [1]:
!pip install tensorflow==2.16.2 tensorflow-metal==1.2.0 scikit-learn matplotlib
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report

## Preparing Dataset

These are just the paths to the dataset and where to save the trained model. I kept them as variables so there's just one point needed to change to train on different datasets.

In [ ]:
PATH_TO_DATA = "../chest_xray/xray_transformed/transform_dataset.csv"
SAVE_PATH = "models/transformed_xray.keras"

# checking if will train on the mac gpu
# print(tf.__version__) 
print(tf.config.list_physical_devices())

Preparing dataset; the csv is read and images are fetched using the paths. Each image is read, resized and normalised. Processed images and labels are stored.

In [ ]:
def load_image(path, label):    #load image and the label from the csv
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img,(224,224))
    img = img/255.0   #normalise img
    return img, label

def make_dataset(df, shuffle=False):
    paths  = df["path"].values
    labels = df["label"].values.astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

df = pd.read_csv(PATH_TO_DATA)

total_train = df[df["split"] != "test"]
test = df[df["split"] == "test"]

train, val = train_test_split(total_train, test_size=0.2, random_state=42, stratify=total_train["label"])

print("train:", train["label"].value_counts().to_dict())
print("val:  ", val["label"].value_counts().to_dict())
print("test: ", test["label"].value_counts().to_dict())

train_ds = make_dataset(train, shuffle=True)
val_ds = make_dataset(val)
test_ds = make_dataset(test)

On initial training, the model was barely training and was performing poorly. I then analysed the class balance in the dataset to guage balance of pneumonia and healthy cases.

In [ ]:

# Compute class weights
train_labels = train["label"].values
total = len(train_labels)
n_normal    = np.sum(train_labels == 0)
n_pneumonia = np.sum(train_labels == 1)

class_weight = {
    0: total / (2 * n_normal),      # NORMAL
    1: total / (2 * n_pneumonia),   # PNEUMONIA
}

print(class_weight)

## Building Model and Training

I built a simple layer CNN followed by a dense network for classification. This model is very simple; I kept it that way for 2 reasons:
- I trained the model locally, so there was a compute limitation
- the classification task itself is not very complex, and so I thought a more complex model would overfit on the train data and it would be hard to truly study dependencies on underlying medical dataset quality

Ideally I would want to train a segmentation model for multiple things which would be complex, and then test it on different datasets; I think that would be a more conclusive study. However, that would also require more data, labels and far more compute. 

Model details:
- 4 layer CNN with a fully connected classification layer
- each inner layer has ReLU activation with sigmoid used at end for classification
- adam optimiser with 0.0001 lr
- uses cross entropy loss

I have also printed model details


In [ ]:
def build_model():
    model = models.Sequential([
        
        # Block 1
        layers.Conv2D(32, (3, 3), padding="same", input_shape=(224, 224, 3)),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2, 2),

        # Block 2
        layers.Conv2D(64, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2, 2),

        # Block 3
        layers.Conv2D(128, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2, 2),

        # Block 4
        layers.Conv2D(256, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(2, 2),

        # Classifier head
        layers.Flatten(),
        layers.BatchNormalization(),
        layers.Dense(256),
        layers.ReLU(),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")]
    )
    return model

model = build_model()
model.summary()

Trained the model on 20 epochs, and saved the model.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight
)
model.save(SAVE_PATH)

Plotting the loss curve for train and validation sets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"],     label="train")
axes[0].plot(history.history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"],     label="train")
axes[1].plot(history.history["val_accuracy"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig("training_transformed.png", dpi=150)
plt.show()

## Evaluating Model on Test Set

Training the model on test set using a confusion matrix; I have also displayed the model performance report below. 

In [1]:
# matrix_label = 'Confusion_Original.png'

print("\n── Test set evaluation ──")
y_true = test["label"].values

y_pred_probs = model.predict(test_ds)
y_pred = (y_pred_probs >= 0.5).astype(int).flatten()

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["NORMAL", "PNEUMONIA"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.savefig('Confusion_Transformed.png', dpi=150)
plt.show()

print(classification_report(y_true, y_pred, target_names=["NORMAL", "PNEUMONIA"]))



── Test set evaluation ──


NameError: name 'test' is not defined